# Medical News RAG — No OpenAI API

This notebook uses **Hugging Face local embeddings + Groq**. No OpenAI credits are required.

### Faster version
- Removes duplicate embedding initialization.
- Reuses the persistent Chroma database instead of re-embedding on every run.
- Uses larger embedding batches for faster CPU processing.
- Retrieves only 3 documents per question.
- Uses the faster Groq `openai/gpt-oss-20b` model.

In [2]:
# Cell 1 — Imports and environment
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
print("GROQ_API_KEY loaded:", bool(groq_api_key))

if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY not found. Create a .env file in the project folder "
        "and add GROQ_API_KEY=your_key"
    )

GROQ_API_KEY loaded: True


## 1. Load the text data

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "./new_articles/",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)

data = loader.load()
print("Documents loaded:", len(data))

if not data:
    raise ValueError("No .txt files found in ./new_articles/")

C:\Users\ELCOT\AppData\Local\Temp\ipykernel_13320\889557537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Documents loaded: 21


## 2. Split the documents into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Slightly larger chunks + smaller overlap = fewer chunks and faster embedding.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(data)
print("Chunks created:", len(chunks))

Chunks created: 180


## 3. Create local Hugging Face embeddings

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={
        "batch_size": 64,
        "normalize_embeddings": True,
    },
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [6]:
# Quick embedding test — run once if you want to verify the model.
test_embedding = embedding.embed_documents(
    ["This is a test sentence for the medical chatbot."]
)

print("Embedding works")
print("Embedding size:", len(test_embedding[0]))

Embedding works
Embedding size: 384


In [7]:
print(type("diabetes.pdf"))
print(len("diabetes.pdf"))

<class 'str'>
12


In [8]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "diabetes.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Documents loaded:", len(documents))

Documents loaded: 23


In [9]:
print(type(documents))
print(documents[0].page_content[:500])

<class 'list'>
Diabetes


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 25


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


: 

## 4. Create Chroma vector database

In [ ]:
from langchain_chroma import Chroma
import os

PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "medical_news"

vectordb = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
    embedding_function=embedding
)

existing_count = vectordb._collection.count()

print("Existing vectors:", existing_count)

if existing_count == 0:

    BATCH_SIZE = 16

    for start in range(0, len(chunks), BATCH_SIZE):

        batch = chunks[start:start + BATCH_SIZE]

        vectordb.add_documents(batch)

        done = min(start + len(batch), len(chunks))

        print(f"Added {done}/{len(chunks)} chunks")

    print("Chroma vector database created successfully.")

else:

    print("Existing Chroma database reused.")
    print("Existing vectors:", existing_count)

print("Database folder:", os.path.abspath(PERSIST_DIR))

: 

### Optional: reset Chroma database

Run this only when you intentionally want to delete and rebuild the vector database.
For normal use, keep `FORCE_REBUILD = False` above.

In [ ]:
# Optional reset cell
import shutil
import gc
import os

gc.collect()
chroma_path = "./chroma_db"

if os.path.exists(chroma_path):
    try:
        shutil.rmtree(chroma_path)
        print("Chroma database deleted successfully. Restart the kernel and run the notebook again.")
    except PermissionError:
        print("Permission denied. Close running Chroma/Jupyter cells and restart the kernel.")
else:
    print("No Chroma database folder found.")

## 5. Test document retrieval

In [ ]:
query = "How much money was Microsoft raised?"

retriever = vectordb.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke(query)

print("Retrieved documents:", len(docs))

for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:500])

## 6. Connect Groq LLM

In [ ]:
from langchain_groq import ChatGroq

# GPT-OSS 20B is used here for lower latency.
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=groq_api_key,
    max_tokens=512,
    reasoning_effort="low",
)

print("Groq LLM loaded successfully.")

In [ ]:
# Test Groq separately before building the RAG chain.
response = llm.invoke("What is Microsoft?")
print(response.content)

## 7. Build the RAG chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

system_prompt = (
    "You are a helpful medical/news document assistant. "
    "Use only the retrieved context to answer the user's question. "
    "If the answer is not present in the context, say: "
    "I don't know based on the provided documents. "
    "Keep the answer concise.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

combine_docs_chain = create_stuff_documents_chain(
    llm,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    combine_docs_chain
)

print("RAG chain created successfully.")

## 8. Ask a question

In [ ]:
query = "How much money was Microsoft raised?"

result = rag_chain.invoke({"input": query})

print("Answer:\n")
print(result["answer"])

print("\nSources:")
seen_sources = set()

for doc in result["context"]:
    source = doc.metadata.get("source")
    if source and source not in seen_sources:
        print("-", source)
        seen_sources.add(source)

## 9. Interactive Medical / News RAG chatbot

In [ ]:
print("Medical / News RAG Chatbot")
print("Type 'exit' to stop.\n")

while True:
    query = input("Input query: ").strip()

    if query.lower() == "exit":
        print("Chatbot stopped.")
        break

    if not query:
        continue

    result = rag_chain.invoke({"input": query})

    print("\nAnswer:")
    print(result["answer"])

    print("\nSources:")
    seen_sources = set()

    for doc in result["context"]:
        source = doc.metadata.get("source")
        if source and source not in seen_sources:
            print("-", source)
            seen_sources.add(source)

    print()